In [ ]:
# DO NOT RUN THIS MORE THAN ONCE. This is a one-time data retrieval script to get historical minute data for TQQQ from Alpaca's free API. 

import pandas as pd
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.enums import DataFeed
from datetime import datetime
import os
from dotenv import load_dotenv

# 1. Setup your free Alpaca API Keys
load_dotenv()
API_KEY = os.environ.get("ALPACA_API_KEY")
SECRET_KEY = os.environ.get("ALPACA_SECRET_KEY")

# Initialize the historical data client
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

# 2. Define the parameters (Alpaca free tier goes back up to 7 years)
request_params = StockBarsRequest(
    symbol_or_symbols="TQQQ",
    timeframe=TimeFrame.Minute,
    start=datetime(2016, 1, 1), # Max out the free historical limit
    end=datetime(2025, 12, 31),
    # CRUCIAL FIX: Tells Alpaca to only return standard 9:30 AM - 4:00 PM data
    feed=DataFeed.SIP
)

print("Fetching historical minute data from Alpaca... This may take a moment.")
# 3. Retrieve the bars
bars = client.get_stock_bars(request_params)

# 4. Convert the results directly into a Pandas DataFrame
df = bars.df

# 5. Clean up multi-index format so it's a standard OHLCV table
df = df.reset_index()
df.set_index('timestamp', inplace=True)

# 6. Save it to a local CSV file right inside your VS Code workspace directory!
df.to_csv("TQQQ_intraday_data.csv")
print("Saved successfully to 'TQQQ_intraday_data.csv'!")

In [2]:
import pandas as pd
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.enums import DataFeed
from datetime import datetime
import os
from dotenv import load_dotenv

In [4]:
# 1. Load your raw file
df = pd.read_csv("TQQQ_intraday_data.csv", parse_dates=['timestamp'], index_col='timestamp')

# 2. Rename columns to match what Backtesting.py expects (Open, High, Low, Close, Volume)
df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})

# 3. Convert UTC (+00:00) directly into New York Time (Eastern Time)
df.index = df.index.tz_convert('America/New_York')

# 4. Strip the timezone text ('-05:00' or '-04:00') so it behaves like standard local time
df.index = df.index.tz_localize(None)

# 5. Filter strictly for US Stock Market Core Hours (09:30 AM to 04:00 PM)
# This throws away any overnight/pre-market noise
df = df.between_time('09:30', '16:00')

# 6. Check your new structure
print(f"Total rows remaining: {len(df)}")
df

Total rows remaining: 779263


,symbol,Open,High,Low,Close,Volume,trade_count,vwap
timestamp,,,,,,,,
2016-01-04 09:30:00,TQQQ,106.65,106.7300,106.0000,106.0800,262807.0,606.0,106.578812
2016-01-04 09:31:00,TQQQ,106.08,106.0830,105.5600,105.8750,35666.0,246.0,105.827157
2016-01-04 09:32:00,TQQQ,105.89,106.3599,105.8301,106.2950,33076.0,155.0,106.162660
2016-01-04 09:33:00,TQQQ,106.27,106.2700,105.8795,106.1352,33882.0,218.0,106.046604
2016-01-04 09:34:00,TQQQ,106.11,106.1100,105.6700,105.8304,22509.0,174.0,105.850556
...,...,...,...,...,...,...,...,...
2023-12-29 15:56:00,TQQQ,50.58,50.6899,50.5800,50.6399,220591.0,757.0,50.629077
2023-12-29 15:57:00,TQQQ,50.63,50.6450,50.5700,50.5766,348458.0,906.0,50.606164
2023-12-29 15:58:00,TQQQ,50.58,50.6400,50.5800,50.6001,260496.0,754.0,50.610019


In [ ]:
# Final (Original)

import pandas as pd
import datetime as dt
from backtesting import Backtest, Strategy


EARLY_CLOSE_DATES = [
    '2017-07-03 12:59:00',
    '2018-07-03 12:59:00',
    '2018-11-23 12:59:00',
    '2019-12-24 12:59:00',
    '2020-11-27 12:59:00',
    '2020-12-24 12:59:00'
]

# 2. create a strategy class
class OpeningRangeBreakout(Strategy):
    open_range_minutes = 5 # length of the opening range
    last_minute_bar_in_opening_range = dt.time(9, 30 + open_range_minutes)
    exit_minute_bar = dt.time(15, 59)
    risk_percent = 0.01
    take_profit_multiple = 10
    max_leverage = 4

    # what do we want to initialize at the beginning
    def init(self):
        self.current_day        = None   # tracks the current day (YYYY-MM-DD)
        self.current_day_open   = None
        self.opening_range_high = None   # opening range high
        self.opening_range_low  = None   # opening range low

    # wait every day is going to have a different opening range high and low
    def _reset_range(self, day, open):
        self.current_day        = day
        self.current_day_open   = open
        self.opening_range_high = None
        self.opening_range_low  = None


    def _get_position_size(self, entry_price: float, stop_price: float) -> int:
        per_share_risk = abs(entry_price - stop_price)  #  |P − S|  =  R

        if per_share_risk == 0:
            return 0

        # Risk-based cap: position that loses 1 % of equity at the stop
        shares_by_risk = (self.risk_percent * self.equity) / per_share_risk

        # Leverage-based cap: shares affordable with 4× buying power
        shares_by_leverage = (self.max_leverage * self.equity) / entry_price

        # Final size: smaller of the two, floored to an int
        return int(min(shares_by_risk, shares_by_leverage))

    def next(self):
        # get the timestamp and extract the date
        t = self.data.index[-1]
        current_bar_date = t.date()

        # detect when the date changes to a new day
        if current_bar_date != self.current_day:
            self._reset_range(current_bar_date, self.data.Open[-1])
            print(f"new date {current_bar_date}")

        # calculate opening range
        if t.time() <= self.last_minute_bar_in_opening_range:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time() < self.last_minute_bar_in_opening_range:
                return  # don’t trade yet

        # Right when the opening range closes, log the opening range high and low for debugging
        if t.time() == self.last_minute_bar_in_opening_range:
          print(f"opening range high is {self.opening_range_high}")
          print(f"opening range low is {self.opening_range_low}")

          if not self.position:
            range_size = self.opening_range_high - self.opening_range_low
            planned_entry_price = self.data.Close[-1]

            if self.data.Close[-1] > self.current_day_open:
                stop_loss_price = self.opening_range_low
                position_size = self._get_position_size(planned_entry_price, stop_loss_price)
                take_profit_price = planned_entry_price + (self.take_profit_multiple * range_size)

                print(f"going long, position size {position_size} at planned price {planned_entry_price}, stop loss {stop_loss_price}")
                order = self.buy(size=position_size, tp=take_profit_price, sl=stop_loss_price)

            elif self.data.Close[-1] < self.current_day_open:
                stop_loss_price = self.opening_range_high
                position_size = self._get_position_size(planned_entry_price, stop_loss_price)
                take_profit_price = planned_entry_price - (self.take_profit_multiple * range_size)

                print(f"going short, position size {position_size} shares at planned price {planned_entry_price}, stop loss {stop_loss_price}")
                order = self.sell(size=position_size, tp=take_profit_price, sl=stop_loss_price)
            else:
              print("doji, doing nothing")


        current_bar_date_time = f"{current_bar_date} {t.time()}"

        # is market is about to close, close my position
        if self.position and t.time() == self.exit_minute_bar or current_bar_date_time in EARLY_CLOSE_DATES:
          print("closing out position")
          self.position.close()



# 1. read a csv of minute data
df = pd.read_csv("TQQQ_intraday_data.csv", parse_dates=['timestamp'], index_col='timestamp')
df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})
df.index = df.index.tz_convert('America/New_York')
df.index = df.index.tz_localize(None)
df = df.between_time('09:30', '16:00')
#df = df.loc['2016-01-05':'2023-12-29']
# df

def per_share_commission(size, price):
    return abs(size) * 0.0005

# 3. create a backtest, pass in data and the strategy you want to run on the data
bt = Backtest(df, OpeningRangeBreakout, cash=25_000, commission=per_share_commission, margin=0.25)

# run the strategy
stats = bt.run()

# plot ths strategy
bt.plot()

print(stats)

new date 2016-01-04
opening range high is 106.3599
opening range low is 105.56
going short, position size 694 shares at planned price 106.0, stop loss 106.3599
new date 2016-01-05
opening range high is 108.93
opening range low is 107.77
going short, position size 222 shares at planned price 107.8201, stop loss 108.93
closing out position
new date 2016-01-06
opening range high is 101.83
opening range low is 100.84
going long, position size 584 at planned price 101.27, stop loss 100.84
closing out position
new date 2016-01-07
opening range high is 96.86
opening range low is 96.05
going long, position size 344 at planned price 96.816, stop loss 96.05
new date 2016-01-08
opening range high is 96.96
opening range low is 95.38
going long, position size 224 at planned price 96.54, stop loss 95.38
new date 2016-01-11
opening range high is 94.03
opening range low is 93.13
going short, position size 297 shares at planned price 93.16, stop loss 94.03
closing out position
new date 2016-01-12
openi

/Users/jonathanneo/LearnQuant/.venv/lib/python3.13/site-packages/backtesting/_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


Start                     2016-01-04 09:30:00
End                       2023-12-29 16:00:00
Duration                   2916 days 06:30:00
Exposure Time [%]                     35.6329
Equity Final [$]                  134687.3969
Equity Peak [$]                   150213.5344
Commissions [$]                      3411.287
Return [%]                          438.74959
Buy & Hold Return [%]               -52.27206
Return (Ann.) [%]                    23.63824
Volatility (Ann.) [%]                 47.3405
CAGR [%]                             15.66459
Sharpe Ratio                          0.49932
Sortino Ratio                         1.77715
Calmar Ratio                           0.6006
Alpha [%]                           437.97469
Beta                                 -0.01482
Max. Drawdown [%]                   -39.35765
Avg. Drawdown [%]                    -1.18428
Max. Drawdown Duration      629 days 00:11:00
Avg. Drawdown Duration        4 days 07:14:00
# Trades                          

In [ ]:
# Modified to align to the logic in the research paper as much as possible

import pandas as pd
import datetime as dt
from backtesting import Backtest, Strategy


EARLY_CLOSE_DATES = [
    '2017-07-03 12:59:00',
    '2018-07-03 12:59:00',
    '2018-11-23 12:59:00',
    '2019-12-24 12:59:00',
    '2020-11-27 12:59:00',
    '2020-12-24 12:59:00'
]

# 2. create a strategy class
class OpeningRangeBreakout(Strategy):
    open_range_minutes = 5 # length of the opening range
    last_minute_bar_in_opening_range = dt.time(9, 30 + open_range_minutes)
    exit_minute_bar = dt.time(15, 59)
    risk_percent = 0.01
    take_profit_multiple = 3
    max_leverage = 4

    # what do we want to initialize at the beginning
    def init(self):
        self.current_day        = None   # tracks the current day (YYYY-MM-DD)
        self.current_day_open   = None
        self.opening_range_high = None   # opening range high
        self.opening_range_low  = None   # opening range low
        self.traded_today = False 

    # wait every day is going to have a different opening range high and low
    def _reset_range(self, day, open):
        self.current_day        = day
        self.current_day_open   = open
        self.opening_range_high = None
        self.opening_range_low  = None
        self.traded_today = False


    def _get_position_size(self, entry_price: float, stop_price: float) -> int:
        per_share_risk = abs(entry_price - stop_price)  #  |P − S|  =  R

        if per_share_risk == 0:
            return 0

        # Risk-based cap: position that loses 1 % of equity at the stop
        shares_by_risk = (self.risk_percent * self.equity) / per_share_risk

        # Leverage-based cap: shares affordable with 4× buying power
        shares_by_leverage = (self.max_leverage * self.equity) / entry_price

        # Final size: smaller of the two, floored to an int
        return int(min(shares_by_risk, shares_by_leverage))

    def next(self):
            t = self.data.index[-1]
            current_bar_date = t.date()

            # 1. Handle New Day Reset
            if current_bar_date != self.current_day:
                self._reset_range(current_bar_date, self.data.Open[-1])
                print(f"new date {current_bar_date}")

            # 2. Build the Opening Range (09:30 to 09:35)
            if t.time() <= self.last_minute_bar_in_opening_range:
                if self.opening_range_high is not None:
                    self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
                else:
                    self.opening_range_high = self.data.High[-1]

                if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
                else:
                    self.opening_range_low = self.data.Low[-1]

                return  # CRUCIAL: Do not evaluate trades during the opening range window

            # 3. TRUE BREAKOUT MONITORING (09:36 AM onwards)
            if not self.position and not self.traded_today:
                range_size = self.opening_range_high - self.opening_range_low
                current_close = self.data.Close[-1]

                # True Long Breakout Condition: Close crosses ABOVE 5-minute High
                if current_close > self.opening_range_high:
                    stop_loss_price = self.opening_range_low
                    take_profit_price = current_close + (self.take_profit_multiple * range_size)

                    # SAFETY CHECK: Only place order if TP is safely above current price
                    if take_profit_price > current_close:
                        position_size = self._get_position_size(current_close, stop_loss_price)
                        print(f"TRUE LONG BREAKOUT! Close ({current_close}) > High ({self.opening_range_high})")
                        self.buy(size=position_size, tp=take_profit_price, sl=stop_loss_price)
                        self.traded_today = True
                    else:
                        print("Long breakout skipped: Take profit price too close to entry.")

                # True Short Breakout Condition: Close crosses BELOW 5-minute Low
                elif current_close < self.opening_range_low:
                    stop_loss_price = self.opening_range_high
                    take_profit_price = current_close - (self.take_profit_multiple * range_size)

                    # SAFETY CHECK: Only place order if TP is safely BELOW current price
                    if take_profit_price < current_close:
                        position_size = self._get_position_size(current_close, stop_loss_price)
                        print(f"TRUE SHORT BREAKOUT! Close ({current_close}) < Low ({self.opening_range_low})")
                        self.sell(size=position_size, tp=take_profit_price, sl=stop_loss_price)
                        self.traded_today = True
                    else:
                        print("Short breakout skipped: Take profit price too close to entry.")

            # 4. End of Day Closeout
            current_bar_date_time = f"{current_bar_date} {t.time()}"
            if self.position and (t.time() == self.exit_minute_bar or current_bar_date_time in EARLY_CLOSE_DATES):
                print("closing out position at end of day")
                self.position.close()
                self.traded_today = False  # reset for the next day

# 1. read a csv of minute data
df = pd.read_csv("TQQQ_intraday_data.csv", parse_dates=['timestamp'], index_col='timestamp')
df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})
df.index = df.index.tz_convert('America/New_York')
df.index = df.index.tz_localize(None)
df = df.between_time('09:30', '16:00')

def per_share_commission(size, price):
    return abs(size) * 0.0005

# 3. create a backtest, pass in data and the strategy you want to run on the data
bt = Backtest(df, OpeningRangeBreakout, cash=25_000, commission=per_share_commission, margin=0.25)

# run the strategy
stats = bt.run()

# plot ths strategy
bt.plot()

print(stats)

new date 2016-01-04
TRUE LONG BREAKOUT! Close (106.51) > High (106.3599)
new date 2016-01-05
TRUE SHORT BREAKOUT! Close (107.64) < Low (107.77)
closing out position at end of day
TRUE SHORT BREAKOUT! Close (106.5) < Low (107.77)
new date 2016-01-06
TRUE LONG BREAKOUT! Close (101.9772) > High (101.83)
new date 2016-01-07
TRUE LONG BREAKOUT! Close (97.1899) > High (96.86)
new date 2016-01-08
TRUE LONG BREAKOUT! Close (97.145) > High (96.96)
new date 2016-01-11
TRUE SHORT BREAKOUT! Close (92.9) < Low (93.13)
new date 2016-01-12
TRUE LONG BREAKOUT! Close (96.1472) > High (95.8299)
new date 2016-01-13
TRUE LONG BREAKOUT! Close (97.12) > High (96.87)
new date 2016-01-14
TRUE SHORT BREAKOUT! Close (85.71) < Low (85.86)
new date 2016-01-15
TRUE LONG BREAKOUT! Close (83.67) > High (83.64)
new date 2016-01-19
TRUE SHORT BREAKOUT! Close (84.9) < Low (84.91)
new date 2016-01-20
TRUE SHORT BREAKOUT! Close (78.22) < Low (78.73)
new date 2016-01-21
TRUE SHORT BREAKOUT! Close (81.221) < Low (81.27)
ne

/Users/jonathanneo/LearnQuant/.venv/lib/python3.13/site-packages/backtesting/_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


Start                     2016-01-04 09:30:00
End                       2023-12-29 16:00:00
Duration                   2916 days 06:30:00
Exposure Time [%]                     46.9989
Equity Final [$]                   76647.8278
Equity Peak [$]                    77993.9757
Commissions [$]                      1180.658
Return [%]                          206.59131
Buy & Hold Return [%]               -52.27206
Return (Ann.) [%]                    15.20938
Volatility (Ann.) [%]                26.25128
CAGR [%]                             10.16521
Sharpe Ratio                          0.57938
Sortino Ratio                          1.2466
Calmar Ratio                          0.49882
Alpha [%]                           206.48693
Beta                                   -0.002
Max. Drawdown [%]                   -30.49082
Avg. Drawdown [%]                    -0.62344
Max. Drawdown Duration      642 days 01:31:00
Avg. Drawdown Duration        3 days 15:41:00
# Trades                          

In [7]:
# Aligned to the logic in the research paper as much as possible and included data from 2016 to 2025

import pandas as pd
import datetime as dt
from backtesting import Backtest, Strategy


EARLY_CLOSE_DATES = [
    '2017-07-03 12:59:00',
    '2018-07-03 12:59:00',
    '2018-11-23 12:59:00',
    '2019-12-24 12:59:00',
    '2020-11-27 12:59:00',
    '2020-12-24 12:59:00'
]

# 2. create a strategy class
class OpeningRangeBreakout(Strategy):
    open_range_minutes = 5 # length of the opening range
    last_minute_bar_in_opening_range = dt.time(9, 30 + open_range_minutes)
    exit_minute_bar = dt.time(15, 59)
    risk_percent = 0.01
    take_profit_multiple = 3
    max_leverage = 4

    # what do we want to initialize at the beginning
    def init(self):
        self.current_day        = None   # tracks the current day (YYYY-MM-DD)
        self.current_day_open   = None
        self.opening_range_high = None   # opening range high
        self.opening_range_low  = None   # opening range low
        self.traded_today = False 

    # wait every day is going to have a different opening range high and low
    def _reset_range(self, day, open):
        self.current_day        = day
        self.current_day_open   = open
        self.opening_range_high = None
        self.opening_range_low  = None
        self.traded_today = False


    def _get_position_size(self, entry_price: float, stop_price: float) -> int:
        per_share_risk = abs(entry_price - stop_price)  #  |P − S|  =  R

        if per_share_risk == 0:
            return 0

        # Risk-based cap: position that loses 1 % of equity at the stop
        shares_by_risk = (self.risk_percent * self.equity) / per_share_risk

        # Leverage-based cap: shares affordable with 4× buying power
        shares_by_leverage = (self.max_leverage * self.equity) / entry_price

        # Final size: smaller of the two, floored to an int
        return int(min(shares_by_risk, shares_by_leverage))

    def next(self):
            t = self.data.index[-1]
            current_bar_date = t.date()

            # 1. Handle New Day Reset
            if current_bar_date != self.current_day:
                self._reset_range(current_bar_date, self.data.Open[-1])
                print(f"new date {current_bar_date}")

            # 2. Build the Opening Range (09:30 to 09:35)
            if t.time() <= self.last_minute_bar_in_opening_range:
                if self.opening_range_high is not None:
                    self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
                else:
                    self.opening_range_high = self.data.High[-1]

                if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
                else:
                    self.opening_range_low = self.data.Low[-1]

                return  # CRUCIAL: Do not evaluate trades during the opening range window

            # 3. TRUE BREAKOUT MONITORING (09:36 AM onwards)
            if not self.position and not self.traded_today:
                range_size = self.opening_range_high - self.opening_range_low
                current_close = self.data.Close[-1]

                # True Long Breakout Condition: Close crosses ABOVE 5-minute High
                if current_close > self.opening_range_high:
                    stop_loss_price = self.opening_range_low # current_close - (self.take_profit_multiple * range_size)
                    take_profit_price = current_close + (self.take_profit_multiple * range_size)

                    # SAFETY CHECK: Only place order if TP is safely above current price
                    if take_profit_price > current_close:
                        position_size = self._get_position_size(current_close, stop_loss_price)
                        print(f"TRUE LONG BREAKOUT! Close ({current_close}) > High ({self.opening_range_high})")
                        self.buy(size=position_size, tp=take_profit_price, sl=stop_loss_price)
                        self.traded_today = True
                    else:
                        print("Long breakout skipped: Take profit price too close to entry.")

                # True Short Breakout Condition: Close crosses BELOW 5-minute Low
                elif current_close < self.opening_range_low:
                    stop_loss_price = self.opening_range_high
                    take_profit_price = current_close - (self.take_profit_multiple * range_size)

                    # SAFETY CHECK: Only place order if TP is safely BELOW current price
                    if take_profit_price < current_close:
                        position_size = self._get_position_size(current_close, stop_loss_price)
                        print(f"TRUE SHORT BREAKOUT! Close ({current_close}) < Low ({self.opening_range_low})")
                        self.sell(size=position_size, tp=take_profit_price, sl=stop_loss_price)
                        self.traded_today = True
                    else:
                        print("Short breakout skipped: Take profit price too close to entry.")

            # 4. End of Day Closeout
            current_bar_date_time = f"{current_bar_date} {t.time()}"
            if self.position and (t.time() == self.exit_minute_bar or current_bar_date_time in EARLY_CLOSE_DATES):
                print("closing out position at end of day")
                self.position.close()
                self.traded_today = False  # reset for the next day

# 1. read a csv of minute data
df = pd.read_csv("TQQQ_intraday_data.csv", parse_dates=['timestamp'], index_col='timestamp')
df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})
df.index = df.index.tz_convert('America/New_York')
df.index = df.index.tz_localize(None)
df = df.between_time('09:30', '16:00')

def per_share_commission(size, price):
    return abs(size) * 0.0005

# 3. create a backtest, pass in data and the strategy you want to run on the data
bt = Backtest(df, OpeningRangeBreakout, cash=25_000, commission=per_share_commission, margin=0.25)

# run the strategy
stats = bt.run()

# plot ths strategy
bt.plot()

print(stats)

new date 2016-01-04
TRUE LONG BREAKOUT! Close (106.51) > High (106.3599)
new date 2016-01-05
TRUE SHORT BREAKOUT! Close (107.64) < Low (107.77)
closing out position at end of day
TRUE SHORT BREAKOUT! Close (106.5) < Low (107.77)
new date 2016-01-06
TRUE LONG BREAKOUT! Close (101.9772) > High (101.83)
new date 2016-01-07
TRUE LONG BREAKOUT! Close (97.1899) > High (96.86)
new date 2016-01-08
TRUE LONG BREAKOUT! Close (97.145) > High (96.96)
new date 2016-01-11
TRUE SHORT BREAKOUT! Close (92.9) < Low (93.13)
new date 2016-01-12
TRUE LONG BREAKOUT! Close (96.1472) > High (95.8299)
new date 2016-01-13
TRUE LONG BREAKOUT! Close (97.12) > High (96.87)
new date 2016-01-14
TRUE SHORT BREAKOUT! Close (85.71) < Low (85.86)
new date 2016-01-15
TRUE LONG BREAKOUT! Close (83.67) > High (83.64)
new date 2016-01-19
TRUE SHORT BREAKOUT! Close (84.9) < Low (84.91)
new date 2016-01-20
TRUE SHORT BREAKOUT! Close (78.22) < Low (78.73)
new date 2016-01-21
TRUE SHORT BREAKOUT! Close (81.221) < Low (81.27)
ne

/Users/jonathanneo/LearnQuant/.venv/lib/python3.13/site-packages/backtesting/_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '1D'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


Start                     2016-01-04 09:30:00
End                       2025-12-30 16:00:00
Duration                   3648 days 06:30:00
Exposure Time [%]                    46.86024
Equity Final [$]                  152760.9227
Equity Peak [$]                   180009.6865
Commissions [$]                      2027.297
Return [%]                          511.04369
Buy & Hold Return [%]                -48.9621
Return (Ann.) [%]                    20.02289
Volatility (Ann.) [%]                27.51059
CAGR [%]                             13.31751
Sharpe Ratio                          0.72782
Sortino Ratio                         1.65389
Calmar Ratio                          0.65669
Alpha [%]                           510.89838
Beta                                 -0.00297
Max. Drawdown [%]                   -30.49082
Avg. Drawdown [%]                    -0.67382
Max. Drawdown Duration      642 days 01:31:00
Avg. Drawdown Duration        3 days 01:55:00
# Trades                          

In [6]:
# 1. Isolate a single trading day where you know a trade happened
df_single_day = df.loc['2024-12-20':'2024-12-20']

# 2. Run a clean micro-backtest for just that day
bt_day = Backtest(df_single_day, OpeningRangeBreakout, cash=25_000, commission=per_share_commission, margin=0.25)
stats_day = bt_day.run()

# 3. Force it to render uncompressed 1-minute data in a full browser tab
bt_day.plot(resample=False, open_browser=True)

# Extract the complete history of closed trades
trade_log = stats['_trades']

# Filter the ledger to only show trades that occurred on your day of interest
specific_day_trades = trade_log[trade_log['EntryTime'].dt.date == pd.to_datetime('2024-12-20').date()]

specific_day_trades

new date 2024-12-20
TRUE LONG BREAKOUT! Close (79.5499) > High (79.2)


,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag
2596,1461,875018,875029,79.5498,78.75,78.75,80.9599,-1169.9688,1.461,-0.010067,2024-12-20 09:39:00,2024-12-20 09:50:00,0 days 00:11:00,None
